In [54]:
import pandas as pd
import numpy as np

# Load validated dataset
df = pd.read_csv("European_Bank.csv")

print("Dataset shape:", df.shape)
df.head()

Dataset shape: (10000, 14)


,Year,CustomerId,Surname,CreditScore,Geography,Gender,Age,Tenure,Balance,NumOfProducts,HasCrCard,IsActiveMember,EstimatedSalary,Exited
0,2025,15634602,Hargrave,619,France,Female,42,2,0.00,1,1,1,101348.88,1
1,2025,15647311,Hill,608,Spain,Female,41,1,83807.86,1,0,1,112542.58,0
2,2025,15619304,Onio,502,France,Female,42,8,159660.80,3,1,0,113931.57,1
3,2025,15701354,Boni,699,France,Female,39,1,0.00,2,0,0,93826.63,0
4,2025,15737888,Mitchell,850,Spain,Female,43,2,125510.82,1,1,1,79084.10,0


In [55]:
# Create a working copy
df_clean = df.copy()

print("Working dataset created.")
df_clean

Working dataset created.


,Year,CustomerId,Surname,CreditScore,Geography,Gender,Age,Tenure,Balance,NumOfProducts,HasCrCard,IsActiveMember,EstimatedSalary,Exited
0,2025,15634602,Hargrave,619,France,Female,42,2,0.00,1,1,1,101348.88,1
1,2025,15647311,Hill,608,Spain,Female,41,1,83807.86,1,0,1,112542.58,0
2,2025,15619304,Onio,502,France,Female,42,8,159660.80,3,1,0,113931.57,1
3,2025,15701354,Boni,699,France,Female,39,1,0.00,2,0,0,93826.63,0
4,2025,15737888,Mitchell,850,Spain,Female,43,2,125510.82,1,1,1,79084.10,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
9995,2025,15606229,Obijiaku,771,France,Male,39,5,0.00,2,1,0,96270.64,0
9996,2025,15569892,Johnstone,516,France,Male,35,10,57369.61,1,1,1,101699.77,0
9997,2025,15584532,Liu,709,France,Female,36,7,0.00,1,0,1,42085.58,1
9998,2025,15682355,Sabbatini,772,Germany,Male,42,3,75075.31,2,1,0,92888.52,1


In [56]:
# Standardize column names

df_clean.columns = (
    df_clean.columns
    .str.strip()
    .str.replace(" ", "_")
)

df_clean.columns

Index(['Year', 'CustomerId', 'Surname', 'CreditScore', 'Geography', 'Gender',
       'Age', 'Tenure', 'Balance', 'NumOfProducts', 'HasCrCard',
       'IsActiveMember', 'EstimatedSalary', 'Exited'],
      dtype='object')

In [57]:
# Clean categorical columns

df_clean["Geography"] = df_clean["Geography"].astype(str).str.strip()
df_clean["Gender"] = df_clean["Gender"].astype(str).str.strip()

print(df_clean["Geography"].unique())
print(df_clean["Gender"].unique())

['France' 'Spain' 'Germany']
['Female' 'Male']


In [58]:
binary_columns = [
    "HasCrCard",
    "IsActiveMember",
    "Exited"
]

for col in binary_columns:
    df_clean[col] = df_clean[col].astype(int)

In [59]:
numeric_columns = [
    "CreditScore",
    "Age",
    "Tenure",
    "Balance",
    "NumOfProducts",
    "EstimatedSalary"
]

for col in numeric_columns:
    df_clean[col] = pd.to_numeric(df_clean[col], errors="coerce")

df_clean[numeric_columns].dtypes

CreditScore          int64
Age                  int64
Tenure               int64
Balance            float64
NumOfProducts        int64
EstimatedSalary    float64
dtype: object

In [60]:
df_clean.isnull().sum()

Year               0
CustomerId         0
Surname            0
CreditScore        0
Geography          0
Gender             0
Age                0
Tenure             0
Balance            0
NumOfProducts      0
HasCrCard          0
IsActiveMember     0
EstimatedSalary    0
Exited             0
dtype: int64

In [61]:
age_bins = [17, 25, 35, 45, 55, 100]

age_labels = [
    "18-25",
    "26-35",
    "36-45",
    "46-55",
    "56+"
]

df_clean["Age_Group"] = pd.cut(
    df_clean["Age"],
    bins=age_bins,
    labels=age_labels
)

df_clean[["Age", "Age_Group"]].head(10)
df_clean["Age_Group"].value_counts().sort_index()

Age_Group
18-25     611
26-35    3542
36-45    3736
46-55    1311
56+       800
Name: count, dtype: int64

In [62]:
df_clean["Balance Rank"] = df_clean["Balance"].rank(method="first")

df_clean["Balance_Segment"] = pd.qcut(
    df_clean["Balance Rank"],
    q=4,
    labels=[
        "Low Balance",
        "Medium Balance",
        "High Balance",
        "Very High Balance"
    ]
)

df_clean.drop(columns=["Balance Rank"], inplace=True)
df_clean["Balance_Segment"].value_counts()

Balance_Segment
Low Balance          2500
Medium Balance       2500
High Balance         2500
Very High Balance    2500
Name: count, dtype: int64

In [63]:
df_clean["Salary Rank"] = df_clean["EstimatedSalary"].rank(method="first")

df_clean["Salary_Segment"] = pd.qcut(
    df_clean["Salary Rank"],
    q=4,
    labels=[
        "Low Salary",
        "Medium Salary",
        "High Salary",
        "Very High Salary"
    ]
)

df_clean.drop(columns=["Salary Rank"], inplace=True)
df_clean["Salary_Segment"].value_counts()

Salary_Segment
Low Salary          2500
Medium Salary       2500
High Salary         2500
Very High Salary    2500
Name: count, dtype: int64

In [64]:
def product_group(products):
    if products == 1:
        return "Single Product"
    elif products == 2:
        return "Two Products"
    else:
        return "3+ Products"

df_clean["Product_Depth_Group"] = df_clean["NumOfProducts"].apply(product_group)
df_clean["Product_Depth_Group"].value_counts()

Product_Depth_Group
Single Product    5084
Two Products      4590
3+ Products        326
Name: count, dtype: int64

In [65]:
df_clean["Engagement_Status"] = np.where(
    df_clean["IsActiveMember"] == 1,
    "Active",
    "Inactive"
)
df_clean["Engagement_Status"].value_counts()

Engagement_Status
Active      5151
Inactive    4849
Name: count, dtype: int64

In [66]:
def engagement_profile(row):
    if row["IsActiveMember"] == 1 and row["NumOfProducts"] >= 2:
        return "Active Engaged"
    
    elif row["IsActiveMember"] == 1 and row["NumOfProducts"] == 1:
        return "Active Low-Product"
    
    elif row["IsActiveMember"] == 0 and row["NumOfProducts"] >= 2:
        return "Inactive Multi-Product"
    
    else:
        return "Inactive Disengaged"

df_clean["Engagement_Profile"] = df_clean.apply(
    engagement_profile,
    axis=1
)
df_clean["Engagement_Profile"].value_counts()

Engagement_Profile
Active Engaged            2588
Active Low-Product        2563
Inactive Disengaged       2521
Inactive Multi-Product    2328
Name: count, dtype: int64

In [67]:
balance_threshold = df_clean["Balance"].quantile(0.75)

print("High balance threshold:", balance_threshold)

High balance threshold: 127644.24


In [68]:
df_clean["High_Value_Customer"] = np.where(
    df_clean["Balance"] >= balance_threshold,
    1,
    0
)
df_clean["High_Value_Customer"].value_counts()

High_Value_Customer
0    7500
1    2500
Name: count, dtype: int64

In [69]:
df_clean["High_Value_Disengaged"] = np.where(
    (df_clean["High_Value_Customer"] == 1) &
    (df_clean["IsActiveMember"] == 0),
    1,
    0
)
df_clean["High_Value_Disengaged"].value_counts()

High_Value_Disengaged
0    8753
1    1247
Name: count, dtype: int64

In [70]:
def customer_value_segment(row):
    
    if row["High_Value_Customer"] == 1 and row["IsActiveMember"] == 1:
        return "High Value - Engaged"
    
    elif row["High_Value_Customer"] == 1 and row["IsActiveMember"] == 0:
        return "High Value - Disengaged"
    
    elif row["High_Value_Customer"] == 0 and row["IsActiveMember"] == 1:
        return "Standard Value - Engaged"
    
    else:
        return "Standard Value - Disengaged"

df_clean["Customer_Value_Segment"] = df_clean.apply(
    customer_value_segment,
    axis=1
)
df_clean["Customer_Value_Segment"].value_counts()

Customer_Value_Segment
Standard Value - Engaged       3898
Standard Value - Disengaged    3602
High Value - Engaged           1253
High Value - Disengaged        1247
Name: count, dtype: int64

In [71]:
df_clean["Credit_Card_Status"] = np.where(
    df_clean["HasCrCard"] == 1,
    "Card Holder",
    "Non Card Holder"
)

In [72]:
df_clean["Engagement_Score"] = (
    df_clean["IsActiveMember"] * 50
    + np.where(df_clean["NumOfProducts"] >= 2, 30, 0)
    + df_clean["HasCrCard"] * 20
)

In [73]:
df_clean["Engagement_Score"].value_counts().sort_index()

Engagement_Score
0       718
20     1803
30      683
50     2433
70     1775
80      756
100    1832
Name: count, dtype: int64

In [74]:
def retention_risk(row):
    
    if row["High_Value_Disengaged"] == 1:
        return "High Risk"
    
    elif row["IsActiveMember"] == 0 and row["NumOfProducts"] == 1:
        return "High Risk"
    
    elif row["IsActiveMember"] == 0:
        return "Medium Risk"
    
    else:
        return "Low Risk"

df_clean["Initial_Retention_Risk"] = df_clean.apply(
    retention_risk,
    axis=1
)
df_clean["Initial_Retention_Risk"].value_counts()

Initial_Retention_Risk
Low Risk       5151
High Risk      2940
Medium Risk    1909
Name: count, dtype: int64

In [75]:
df_clean["Balance_to_Salary_Ratio"] = (
    df_clean["Balance"] /
    df_clean["EstimatedSalary"].replace(0, np.nan)
)
df_clean["Balance_to_Salary_Ratio"].describe()

count    10000.000000
mean         3.878703
std        108.337260
min          0.000000
25%          0.000000
50%          0.747002
75%          1.514022
max      10614.655440
Name: Balance_to_Salary_Ratio, dtype: float64

In [76]:
balance_median = df_clean["Balance"].median()
salary_median = df_clean["EstimatedSalary"].median()

def financial_segment(row):
    
    if row["Balance"] >= balance_median and row["EstimatedSalary"] >= salary_median:
        return "High Salary - High Balance"
    
    elif row["Balance"] >= balance_median and row["EstimatedSalary"] < salary_median:
        return "Low Salary - High Balance"
    
    elif row["Balance"] < balance_median and row["EstimatedSalary"] >= salary_median:
        return "High Salary - Low Balance"
    
    else:
        return "Low Salary - Low Balance"

df_clean["Financial_Segment"] = df_clean.apply(
    financial_segment,
    axis=1
)

In [77]:
df_clean["Churn_Status"] = np.where(
    df_clean["Exited"] == 1,
    "Churned",
    "Retained"
)

In [78]:
new_columns = [
    "Age_Group",
    "Balance_Segment",
    "Salary_Segment",
    "Product_Depth_Group",
    "Engagement_Status",
    "Engagement_Profile",
    "High_Value_Customer",
    "High_Value_Disengaged",
    "Customer_Value_Segment",
    "Credit_Card_Status",
    "Engagement_Score",
    "Initial_Retention_Risk",
    "Balance_to_Salary_Ratio",
    "Financial_Segment",
    "Churn_Status"
]

df_clean[new_columns].head(10)

,Age_Group,Balance_Segment,Salary_Segment,Product_Depth_Group,Engagement_Status,Engagement_Profile,High_Value_Customer,High_Value_Disengaged,Customer_Value_Segment,Credit_Card_Status,Engagement_Score,Initial_Retention_Risk,Balance_to_Salary_Ratio,Financial_Segment,Churn_Status
0,36-45,Low Balance,High Salary,Single Product,Active,Active Low-Product,0,0,Standard Value - Engaged,Card Holder,70,Low Risk,0.000000,High Salary - Low Balance,Churned
1,36-45,Medium Balance,High Salary,Single Product,Active,Active Low-Product,0,0,Standard Value - Engaged,Non Card Holder,50,Low Risk,0.744677,High Salary - Low Balance,Retained
2,36-45,Very High Balance,High Salary,3+ Products,Inactive,Inactive Multi-Product,1,1,High Value - Disengaged,Card Holder,50,High Risk,1.401375,High Salary - High Balance,Churned
3,36-45,Low Balance,Medium Salary,Two Products,Inactive,Inactive Multi-Product,0,0,Standard Value - Disengaged,Non Card Holder,30,Medium Risk,0.000000,Low Salary - Low Balance,Retained
4,36-45,High Balance,Medium Salary,Single Product,Active,Active Low-Product,0,0,Standard Value - Engaged,Card Holder,70,Low Risk,1.587055,Low Salary - High Balance,Retained
5,36-45,High Balance,Very High Salary,Two Products,Inactive,Inactive Multi-Product,0,0,Standard Value - Disengaged,Card Holder,50,Medium Risk,0.759604,High Salary - High Balance,Churned
6,46-55,Low Balance,Low Salary,Two Products,Active,Active Engaged,0,0,Standard Value - Engaged,Card Holder,100,Low Risk,0.000000,Low Salary - Low Balance,Retained
7,26-35,High Balance,High Salary,3+ Products,Inactive,Inactive Multi-Product,0,0,Standard Value - Disengaged,Card Holder,50,Medium Risk,0.963969,High Salary - High Balance,Churned
8,36-45,Very High Balance,Medium Salary,Two Products,Active,Active Engaged,1,0,High Value - Engaged,Non Card Holder,80,Low Risk,1.895518,Low Salary - High Balance,Retained
9,26-35,Very High Balance,Medium Salary,Single Product,Active,Active Low-Product,1,0,High Value - Engaged,Card Holder,70,Low Risk,1.876647,Low Salary - High Balance,Retained


In [79]:
print("Original columns:", len(df.columns))
print("New columns:", len(df_clean.columns))

df_clean.info()

Original columns: 14
New columns: 29
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 10000 entries, 0 to 9999
Data columns (total 29 columns):
 #   Column                   Non-Null Count  Dtype   
---  ------                   --------------  -----   
 0   Year                     10000 non-null  int64   
 1   CustomerId               10000 non-null  int64   
 2   Surname                  10000 non-null  object  
 3   CreditScore              10000 non-null  int64   
 4   Geography                10000 non-null  object  
 5   Gender                   10000 non-null  object  
 6   Age                      10000 non-null  int64   
 7   Tenure                   10000 non-null  int64   
 8   Balance                  10000 non-null  float64 
 9   NumOfProducts            10000 non-null  int64   
 10  HasCrCard                10000 non-null  int64   
 11  IsActiveMember           10000 non-null  int64   
 12  EstimatedSalary          10000 non-null  float64 
 13  Exited                   

In [80]:
# Check missing values in engineered features

engineered_columns = new_columns

print(
    df_clean[engineered_columns]
    .isnull()
    .sum()
)

Age_Group                  0
Balance_Segment            0
Salary_Segment             0
Product_Depth_Group        0
Engagement_Status          0
Engagement_Profile         0
High_Value_Customer        0
High_Value_Disengaged      0
Customer_Value_Segment     0
Credit_Card_Status         0
Engagement_Score           0
Initial_Retention_Risk     0
Balance_to_Salary_Ratio    0
Financial_Segment          0
Churn_Status               0
dtype: int64


In [81]:
# Check duplicate customers again

print("Duplicate Customer IDs:",
      df_clean["CustomerId"].duplicated().sum())

Duplicate Customer IDs: 0


In [82]:
# Check churn values

print("Churn values:",
      df_clean["Exited"].unique())

Churn values: [1 0]


In [83]:
output_file = "European_Bank_Cleaned_Featured.csv"

df_clean.to_csv(
    output_file,
    index=False
)

print(f"✅ Cleaned dataset saved as: {output_file}")

✅ Cleaned dataset saved as: European_Bank_Cleaned_Featured.csv
